In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [3]:
from blimpy import Waterfall

file_path = "../data/raw/spliced_blc0001020304050607_guppi_57635_35657_Gj144_0005.gpuspec.0002.h5"

obs = Waterfall(file_path)

obs.info()

/Users/majorslammage/Desktop/seti_signal_detection/.venv/lib/python3.14/site-packages/blimpy/__init__.py:21: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound



--- File Info ---
DIMENSION_LABELS :   ['frequency' 'feed_id' 'time']
        az_start :                              0.0
       data_type :                                1
            fch1 :                2802.83203125 MHz
            foff :         -0.00286102294921875 MHz
      machine_id :                               20
           nbits :                               32
          nchans :                           351232
            nifs :                                1
     source_name :                            Gj144
         src_dej :                     -9:13:11.007
         src_raj :                       3:32:54.68
    telescope_id :                                6
           tsamp :               1.0737418239999998
   tstart (ISOT) :          2016-09-04T09:54:17.000
    tstart (MJD) :                57635.41269675926
        za_start :                              0.0

Num ints in file :                              273
      File shape :       (273, np.int64(1), 

In [4]:
def extract_signal(obs):
    signal = obs.data[:,0,:]
    return signal

In [5]:
signal = extract_signal(obs)
print(signal.shape)

(273, 351232)


In [6]:
def compute_mean_power(signal):
    mean_power = np.mean(signal, axis=0)
    return mean_power

In [7]:
mean_power = compute_mean_power(signal)
print(mean_power.shape)


(351232,)


In [8]:
def extract_frequencies(obs):
    freqs, _ = obs.grab_data()
    return freqs

In [9]:
freqs = extract_frequencies(obs)

print(freqs.shape)
print(freqs[:5])

(351232,)
[2802.83203125 2802.82917023 2802.8263092  2802.82344818 2802.82058716]


In [10]:
def extract_frequency_window(freqs, mean_power, start_freq, end_freq):
    # create mask
    mask = (freqs >= start_freq) & (freqs <= end_freq)

    # apply mask to freqs
    zoom_freqs = freqs[mask]

    # apply same mask to mean_power
    zoom_power = mean_power[mask]
    
    # return both
    return (zoom_freqs, zoom_power)

In [11]:
zoom_freqs, zoom_power = extract_frequency_window(freqs, mean_power, 2200, 2205)
print(zoom_freqs.shape)
print(zoom_power.shape)

print(zoom_freqs.min())
print(zoom_freqs.max())

(1748,)
(1748,)
2200.0001907348633
2204.9983978271484


In [12]:
def calculate_z_score(zoom_power):
    mean_zoom_power = np.mean(zoom_power)
    std_zoom_power = np.std(zoom_power)

    z_scores = (zoom_power - mean_zoom_power) / std_zoom_power

    return z_scores

In [13]:
z_scores = calculate_z_score(zoom_power)
print(z_scores.shape)
print("Highest z-score:", np.max(z_scores))
print("Lowest z-score:", np.min(z_scores))

(1748,)
Highest z-score: 4.45076
Lowest z-score: -3.0429387


In [14]:
def compute_z_mask(z_scores, threshold):
    z_mask = np.abs(z_scores) >= threshold
    return z_mask

In [15]:
z_mask = compute_z_mask(z_scores, 3)

print(np.sum(z_mask))

11


In [16]:
def calculate_anomaly_candidates(zoom_freqs, zoom_power, z_scores, z_mask):
    anomaly_freqs = zoom_freqs[z_mask]
    anomaly_power = zoom_power[z_mask]
    anomaly_z = z_scores[z_mask]

    return anomaly_freqs, anomaly_power, anomaly_z

In [17]:
anomaly_freqs, anomaly_power, anomaly_z = calculate_anomaly_candidates(zoom_freqs, zoom_power, z_scores, z_mask)

print(anomaly_freqs)
print(anomaly_power)
print(anomaly_z)

[2203.7109375  2202.27184296 2202.26898193 2202.26612091 2202.26325989
 2202.26039886 2202.25753784 2202.25467682 2202.2518158  2202.24895477
 2200.78125   ]
[8.4321336e+09 2.6391721e+09 2.6302300e+09 2.6284925e+09 2.6205814e+09
 2.6135137e+09 2.6156119e+09 2.6062190e+09 2.6042284e+09 2.6070702e+09
 9.1810703e+09]
[ 3.5974162 -3.0031238 -3.0133123 -3.015292  -3.024306  -3.0323591
 -3.0299683 -3.0406706 -3.0429387 -3.0397007  4.45076  ]


In [18]:
def get_channel_spacing(obs):
    channel_spacing = abs(obs.header["foff"])
    return channel_spacing

In [19]:
channel_spacing = get_channel_spacing(obs)
print(channel_spacing)

0.00286102294921875


In [20]:
def get_breaks(anomaly_freqs, channel_spacing, tolerance):
    freqs_diff = np.abs(np.diff(anomaly_freqs))
    breaks = freqs_diff > channel_spacing * tolerance
    return breaks

In [21]:
breaks = get_breaks(anomaly_freqs, channel_spacing, 1.5)
print(breaks)

[ True False False False False False False False False  True]


In [22]:
def get_split_indices(breaks):
    split_indices = np.where(breaks)[0] + 1
    return split_indices

In [23]:
split_indices = get_split_indices(breaks)

print(split_indices)

[ 1 10]


In [24]:
def get_splits(anomaly_freqs, anomaly_z, split_indices):
    features_freqs = np.split(anomaly_freqs, split_indices)
    features_z = np.split(anomaly_z, split_indices)
    return features_freqs, features_z

In [25]:
features_freqs, features_z = get_splits(anomaly_freqs, anomaly_z, split_indices)

print(features_freqs)
print(features_z)

[array([2203.7109375]), array([2202.27184296, 2202.26898193, 2202.26612091, 2202.26325989,
       2202.26039886, 2202.25753784, 2202.25467682, 2202.2518158 ,
       2202.24895477]), array([2200.78125])]
[array([3.5974162], dtype=float32), array([-3.0031238, -3.0133123, -3.015292 , -3.024306 , -3.0323591,
       -3.0299683, -3.0406706, -3.0429387, -3.0397007], dtype=float32), array([4.45076], dtype=float32)]


In [26]:
def get_candidate_records(features_freqs, features_z):
    candidate_records = []

    for freqs, z in zip(features_freqs, features_z):
        channel_count = len(freqs)
        center_frequency = np.mean(freqs)
        frequency_span = max(freqs) - min(freqs)
        strongest_z_index = np.argmax(abs(z))
        max_z = z[strongest_z_index]
        signed_strongest_z = "Negative" if max_z < 0 else "Positive"

        candidate_records.append({"channel_count":channel_count,
                                "center_frequency":center_frequency,
                                "frequency_span":frequency_span,
                                "strongest_z":max_z,
                                "signed_strongest_z":signed_strongest_z
                                })
    
    return candidate_records

In [27]:
candidate_records = get_candidate_records(features_freqs, features_z)

print(candidate_records)

[{'channel_count': 1, 'center_frequency': np.float64(2203.7109375), 'frequency_span': np.float64(0.0), 'strongest_z': np.float32(3.5974162), 'signed_strongest_z': 'Positive'}, {'channel_count': 9, 'center_frequency': np.float64(2202.260398864746), 'frequency_span': np.float64(0.02288818359375), 'strongest_z': np.float32(-3.0429387), 'signed_strongest_z': 'Negative'}, {'channel_count': 1, 'center_frequency': np.float64(2200.78125), 'frequency_span': np.float64(0.0), 'strongest_z': np.float32(4.45076), 'signed_strongest_z': 'Positive'}]


In [28]:
def candidate_record_pipeline(
    obs, 
    start_freq,
    end_freq,
    z_threshold = 3,
    tolerance = 1.5
):
    signal = extract_signal(obs)
    mean_power = compute_mean_power(signal)
    freqs = extract_frequencies(obs)
    zoom_freqs, zoom_power = extract_frequency_window(freqs, mean_power, start_freq, end_freq)
    z_scores = calculate_z_score(zoom_power)
    z_mask = compute_z_mask(z_scores, z_threshold)
    if not any(z_mask):
        return []
    anomaly_freqs, anomaly_power, anomaly_z = calculate_anomaly_candidates(zoom_freqs, zoom_power, z_scores, z_mask)
    channel_spacing = get_channel_spacing(obs)
    breaks = get_breaks(anomaly_freqs, channel_spacing, tolerance)
    split_indices = get_split_indices(breaks)
    features_freqs, features_z = get_splits(anomaly_freqs, anomaly_z, split_indices)
    candidate_records = get_candidate_records(features_freqs, features_z)

    return candidate_records



In [29]:
candidate_records = candidate_record_pipeline(
    obs,
    2200,
    2205
)

print(candidate_records)

[{'channel_count': 1, 'center_frequency': np.float64(2203.7109375), 'frequency_span': np.float64(0.0), 'strongest_z': np.float32(3.5974162), 'signed_strongest_z': 'Positive'}, {'channel_count': 9, 'center_frequency': np.float64(2202.260398864746), 'frequency_span': np.float64(0.02288818359375), 'strongest_z': np.float32(-3.0429387), 'signed_strongest_z': 'Negative'}, {'channel_count': 1, 'center_frequency': np.float64(2200.78125), 'frequency_span': np.float64(0.0), 'strongest_z': np.float32(4.45076), 'signed_strongest_z': 'Positive'}]


In [30]:
def generate_windows(freqs, window_size):
    min_freq = min(freqs)
    max_freq = max(freqs)

    start_freq = min_freq

    windows = []

    while start_freq + window_size <= max_freq:
        windows.append((start_freq, start_freq + window_size))
        start_freq += window_size
    
    if start_freq < max_freq:
        windows.append((start_freq, max_freq))

    return windows

In [31]:
windows = generate_windows(freqs, 5)

print("Number of windows:", len(windows))
print("First 5:", windows[:5])
print("Last 5:", windows[-5:])

Number of windows: 201
First 5: [(np.float64(1797.9520797729492), np.float64(1802.9520797729492)), (np.float64(1802.9520797729492), np.float64(1807.9520797729492)), (np.float64(1807.9520797729492), np.float64(1812.9520797729492)), (np.float64(1812.9520797729492), np.float64(1817.9520797729492)), (np.float64(1817.9520797729492), np.float64(1822.9520797729492))]
Last 5: [(np.float64(2777.952079772949), np.float64(2782.952079772949)), (np.float64(2782.952079772949), np.float64(2787.952079772949)), (np.float64(2787.952079772949), np.float64(2792.952079772949)), (np.float64(2792.952079772949), np.float64(2797.952079772949)), (np.float64(2797.952079772949), np.float64(2802.83203125))]


In [32]:
all_candidate_records = []

for start_freq, end_freq in windows:
    candidate_records = candidate_record_pipeline(obs, start_freq= start_freq, end_freq=end_freq)
    all_candidate_records.extend(candidate_records)
    

In [33]:
print(len(all_candidate_records))

351


In [34]:
candidate_df = pd.DataFrame(all_candidate_records)

In [35]:
candidate_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 351 entries, 0 to 350
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   channel_count       351 non-null    int64  
 1   center_frequency    351 non-null    float64
 2   frequency_span      351 non-null    float64
 3   strongest_z         351 non-null    float32
 4   signed_strongest_z  351 non-null    str    
dtypes: float32(1), float64(2), int64(1), str(1)
memory usage: 12.5 KB


In [36]:
candidate_df.describe()

,channel_count,center_frequency,frequency_span,strongest_z
count,351.000000,351.000000,351.000000,351.000000
mean,11.481481,2251.976325,0.029988,9.535081
std,24.155900,294.341720,0.069111,11.575318
min,1.000000,1799.414062,0.000000,-4.285960
25%,1.000000,2002.252722,0.000000,3.365071
50%,1.000000,2209.570312,0.000000,6.077146
75%,1.000000,2535.497332,0.000000,14.947170
max,123.000000,2801.367188,0.349045,41.706497


In [37]:
candidate_df['signed_strongest_z'].value_counts()

signed_strongest_z
Positive    278
Negative     73
Name: count, dtype: int64

Exploring channel_count because min, 25%, 50%, 75% are 1 and max is 123, possibly skewing mean

In [38]:
candidate_df["channel_count"].plot.box()
plt.show()

/var/folders/96/b9ghc6nj60952p_5_s5gk7980000gn/T/ipykernel_8265/3963028241.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [39]:
candidate_df['channel_count'].value_counts().sort_index()

channel_count
1      264
2        4
3        6
4        2
6        3
7        1
9        2
11       4
12       3
13       1
14       1
15       1
17       1
19       1
23       2
24       1
25       1
26       1
27       1
28       2
31       1
32       2
33       2
40       1
42       1
46       1
50       1
51       1
52       1
55       1
56       1
58       1
59       1
60       1
64       1
68       1
69       2
71       1
72       2
73       1
75       2
76       4
77       3
79       3
80       2
81       1
82       2
83       2
84       2
85       3
123      1
Name: count, dtype: int64

In [40]:
candidate_df["channel_count"].hist(bins=20)
plt.show()

/var/folders/96/b9ghc6nj60952p_5_s5gk7980000gn/T/ipykernel_8265/3347972005.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [41]:
candidate_df["strongest_z"].hist(bins=20)
plt.show()

/var/folders/96/b9ghc6nj60952p_5_s5gk7980000gn/T/ipykernel_8265/2939548058.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [49]:
def process_observation(obs, window_size=5, z_threshold=3, tolerance=1.5):
    all_candidate_records = []

    freqs = extract_frequencies(obs)
    windows = generate_windows(freqs, window_size)

    for start_freq, end_freq in windows:
        candidate_records = candidate_record_pipeline(
            obs,
            start_freq=start_freq,
            end_freq=end_freq,
            z_threshold=z_threshold,
            tolerance=tolerance
        )

        all_candidate_records.extend(candidate_records)

    return all_candidate_records



In [50]:
all_candidate_records = process_observation(obs)

candidate_df = pd.DataFrame(all_candidate_records)

In [51]:
candidate_df.shape

(351, 5)